# MedScan — Medicine Parsing & Validation Laboratory
> **Purpose:** Production-grade medical entity normalization, parsing validation, and search-infrastructure preparation.
>
> **Phase:** Exploration · Validation · Medical-NLP Stabilisation
>
> **Owner:** MedScan Engineering — Healthcare Intelligence Platform

## 0 · Environment Setup

In [1]:
import pandas as pd
import numpy as np
import re, json, uuid, logging, warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("medscan")

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT       = Path(r"c:/Users/arjun/OneDrive/Desktop/project/Medi Cam")
DATA_RAW   = ROOT / "data" / "raw"
DATA_FINAL = ROOT / "data" / "final"
REPORTS    = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

# ── Config ─────────────────────────────────────────────────────────────────────
INPUT_FILE = DATA_FINAL / "medicine_final_clean.csv"

# Single-token words that almost always mean the ingredient name was truncated
TRUNCATED_FLAGS = {
    "acid","proxetil","sodium","potassium","ester","oxide",
    "sulfate","sulphate","nitrate","phosphate","chloride","acetate",
    "tartrate","gluconate","citrate","maleate","fumarate","benzoate",
    "valerate","propionate","bromide","iodide","lactate","stearate",
}

# Canonical unit map  (regex → normalised label)
UNIT_MAP = {
    r"milligrams?|mg\.?":         "mg",
    r"micrograms?|mcg|μg|µg":      "mcg",
    r"grams?|gm\.?|gms\.?":      "g",
    r"international\s+units?|iu":  "IU",
    r"millilitres?|ml\.?":         "ml",
    r"litres?|\bl\b":             "L",
    r"units?|u\.?":                "units",
    r"milli?eq|meq":               "mEq",
    r"%":                           "percent",
}

VALID_UNITS = {"mg","mcg","g","IU","ml","L","units","mEq","percent"}

log.info("Environment ready.  ROOT = %s", ROOT)

INFO | Environment ready.  ROOT = c:\Users\arjun\OneDrive\Desktop\project\Medi Cam


## 1 · Data Loading

In [2]:
log.info("Loading: %s", INPUT_FILE)
df_raw = pd.read_csv(INPUT_FILE, low_memory=False)
df_raw.columns = df_raw.columns.str.strip().str.lower().str.replace(" ","_",regex=False)

log.info("Loaded  %d rows × %d columns", *df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head(3)

INFO | Loading: c:\Users\arjun\OneDrive\Desktop\project\Medi Cam\data\final\medicine_final_clean.csv
INFO | Loaded  226709 rows × 8 columns


Columns: ['url', 'name', 'composition', 'uses', 'side_effects', 'image_url', 'marketer', 'composition_parsed']


,url,name,composition,uses,side_effects,image_url,marketer,composition_parsed
0,https://www.1mg.com/drugs/cardiomart-h-40mg-12...,Cardiomart H 40mg/12.5mg Tablet,Telmisartan (40mg) + Hydrochlorothiazide (12.5mg),Treatment of Hypertension (high blood pressure...,Most side effects do not require any medical a...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow...",10 Drug Mart,"[('Telmisartan', '40mg'), ('Hydrochlorothiazid..."
1,https://www.1mg.com/drugs/rosumart-10mg-tablet...,Rosumart 10mg Tablet,Rosuvastatin (10mg),Treatment of High cholesterol Treatment of Hig...,Most side effects do not require any medical a...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow...",10 Drug Mart,"[('Rosuvastatin', '10mg')]"
2,https://www.1mg.com/drugs/tendonil-p-100mg-325...,Tendonil P 100mg/325mg Tablet,Aceclofenac (100mg) + Paracetamol (325mg),Pain relief In Pain relief Tendonil P 100mg/32...,Most side effects do not require any medical a...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow...",10 Drug Mart,"[('Aceclofenac', '100mg'), ('Paracetamol', '32..."


## 2 · Dataset Profiling

In [3]:
def profile_dataset(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    rows = []
    for col in df.columns:
        null_n = int(df[col].isna().sum())
        rows.append({
            "column":   col,
            "dtype":    str(df[col].dtype),
            "nulls":    null_n,
            "null_pct": round(null_n / n * 100, 2),
            "unique":   int(df[col].nunique()),
            "sample":   str(df[col].dropna().iloc[0]) if null_n < n else "ALL NULL",
        })
    return pd.DataFrame(rows).set_index("column")

profile_df = profile_dataset(df_raw)
profile_df

,dtype,nulls,null_pct,unique,sample
column,,,,,
url,object,0,0.00,226709,https://www.1mg.com/drugs/cardiomart-h-40mg-12...
name,object,1,0.00,223219,Cardiomart H 40mg/12.5mg Tablet
composition,object,1,0.00,13214,Telmisartan (40mg) + Hydrochlorothiazide (12.5mg)
uses,object,23,0.01,223390,Treatment of Hypertension (high blood pressure...
side_effects,object,0,0.00,197056,Most side effects do not require any medical a...
image_url,object,1,0.00,1,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow..."
marketer,object,0,0.00,12375,10 Drug Mart
composition_parsed,object,0,0.00,7686,"[('Telmisartan', '40mg'), ('Hydrochlorothiazid..."


## 3 · Null Analysis

In [4]:
null_report = (
    df_raw.isnull().sum()
    .rename("nulls")
    .to_frame()
    .assign(null_pct=lambda x: (x["nulls"] / len(df_raw) * 100).round(2))
    .sort_values("nulls", ascending=False)
    .query("nulls > 0")
)
print(null_report.to_string())

CRITICAL = ["name","composition","marketer"]
for col in CRITICAL:
    if col in df_raw.columns:
        n = df_raw[col].isna().sum()
        level = "WARNING" if n > 0 else "INFO"
        log.info("[%s] '%s'  missing=%d", level, col, n)

INFO | [WARNING] 'name'  missing=1
INFO | [WARNING] 'composition'  missing=1
INFO | [INFO] 'marketer'  missing=0


             nulls  null_pct
uses            23      0.01
name             1      0.00
composition      1      0.00
image_url        1      0.00


## 4 · Duplicate Detection

In [5]:
# Detect which column is the primary key URL column
url_col  = next((c for c in ["url","medicine_url"] if c in df_raw.columns), None)
name_col = next((c for c in ["name","medicine_name"] if c in df_raw.columns), None)
comp_col = next((c for c in ["composition","salt_composition"] if c in df_raw.columns), None)

log.info("Key columns — url:%s | name:%s | composition:%s", url_col, name_col, comp_col)

if url_col:
    dup_url = df_raw.duplicated(subset=[url_col], keep="first").sum()
    log.info("Duplicate URLs: %d", dup_url)
    df = df_raw.drop_duplicates(subset=[url_col], keep="first").copy()
else:
    df = df_raw.copy()

if name_col and comp_col:
    dup_nc = df.duplicated(subset=[name_col, comp_col], keep="first").sum()
    log.info("Duplicate name+composition: %d", dup_nc)
    df = df.drop_duplicates(subset=[name_col, comp_col], keep="first")

log.info("Working dataset after dedup: %d rows", len(df))

INFO | Key columns — url:url | name:name | composition:composition
INFO | Duplicate URLs: 0
INFO | Duplicate name+composition: 3065
INFO | Working dataset after dedup: 223644 rows


## 5 · Composition Cleaning

Normalise raw strings before parsing.

In [6]:
def clean_composition(text: str) -> str:
    """Standardise separators, remove boilerplate parenthetical equivalences."""
    if not isinstance(text, str) or not text.strip():
        return ""
    s = text.strip()
    # Standardise separators
    s = re.sub(r"\s*\+\s*", " + ", s)
    s = re.sub(r"\s*/\s*",  " / ", s)
    # Remove pharmacokinetic parentheticals:  (eq. to 250mg base)  (as free base)
    s = re.sub(r"\((?:eq\.?\s*to|as free)[^)]*\)", "", s, flags=re.I)
    # Un-glue bracket dosages:   Paracetamol(500mg)  →  Paracetamol 500mg
    s = re.sub(r"([A-Za-z])\(",  r"\1 (", s)
    s = re.sub(r"\((\d)",       r"\1",   s)
    s = re.sub(r"\)",            "",      s)
    # Collapse whitespace
    s = re.sub(r"\s{2,}", " ", s).strip()
    return s

if comp_col:
    df["composition_clean"] = df[comp_col].apply(clean_composition)
else:
    df["composition_clean"] = ""

print(df[[comp_col, "composition_clean"]].head(10).to_string())

                                                                                               composition                                                                                        composition_clean
0                                                        Telmisartan (40mg) + Hydrochlorothiazide (12.5mg)                                                            Telmisartan 40mg + Hydrochlorothiazide 12.5mg
1                                                                                      Rosuvastatin (10mg)                                                                                        Rosuvastatin 10mg
2                                                                Aceclofenac (100mg) + Paracetamol (325mg)                                                                    Aceclofenac 100mg + Paracetamol 325mg
3                                                   Cefpodoxime Proxetil (200mg) + Clavulanic Acid (125mg)                                              

## 6 · Unit Normalisation

In [7]:
def normalize_unit(raw: str) -> str:
    if not isinstance(raw, str): return ""
    u = raw.strip()
    for pattern, label in UNIT_MAP.items():
        if re.fullmatch(pattern, u, flags=re.I):
            return label
    return u  # return as-is; validator will flag unknown units

# ── Self-test ──────────────────────────────────────────────────────────────────
_unit_tests = [("MG","mg"),("Milligram","mg"),("µg","mcg"),("gm","g"),
               ("IU","IU"),("ml.","ml"),("units","units"),("%","percent")]
all_pass = True
for raw, expected in _unit_tests:
    got = normalize_unit(raw)
    ok  = got == expected
    all_pass = all_pass and ok
    print(f"{'✓' if ok else '✗'}  {raw!r:15s} → {got!r}  (expected {expected!r})")
print("\nUnit normalizer:", "✓ ALL PASS" if all_pass else "✗ FAILURES DETECTED")

✓  'MG'            → 'mg'  (expected 'mg')
✓  'Milligram'     → 'mg'  (expected 'mg')
✓  'µg'            → 'mcg'  (expected 'mcg')
✓  'gm'            → 'g'  (expected 'g')
✓  'IU'            → 'IU'  (expected 'IU')
✓  'ml.'           → 'ml'  (expected 'ml')
✓  'units'         → 'units'  (expected 'units')
✓  '%'             → 'percent'  (expected 'percent')

Unit normalizer: ✓ ALL PASS


## 7 · Ingredient Extraction

**Critical requirement:** multi-word names must be preserved in full.

| Input | Must produce |
|---|---|
| `Cefpodoxime Proxetil 200mg` | `ingredient: "Cefpodoxime Proxetil"` |
| `Clavulanic Acid 125mg` | `ingredient: "Clavulanic Acid"` |
| `Vitamin C / Zinc` | two ingredients, no strength |

In [8]:
# Regex that splits at the first digit boundary following at least one letter word
_SPLIT_PAT = re.compile(
    r"^(.*?[A-Za-z])"          # ingredient name (greedy but must contain a letter)
    r"\s+"                     # separator
    r"(\d+(?:\.\d+)?)"       # numeric strength
    r"\s*(\S*)"               # optional unit
    r"\s*$"
)

def _parse_block(block: str) -> dict:
    """Parse one ingredient block (no + or /)."""
    block = block.strip()
    if not block:
        return None

    m = _SPLIT_PAT.match(block)
    if m:
        ingredient_raw = m.group(1).strip()
        strength       = float(m.group(2))
        unit           = normalize_unit(m.group(3)) if m.group(3) else ""
    else:
        # No numeric strength found — just record the name
        ingredient_raw = block
        strength, unit = None, ""

    flag = None
    words = ingredient_raw.split()
    if len(words) == 1 and words[0].lower() in TRUNCATED_FLAGS:
        flag = "SUSPICIOUS_TRUNCATION"

    return {"ingredient": ingredient_raw, "strength": strength, "unit": unit, "flag": flag}


def parse_composition(comp_clean: str) -> list[dict]:
    """Split on + or / then parse each block."""
    if not isinstance(comp_clean, str) or not comp_clean.strip():
        return []
    blocks = re.split(r"\s*[+/]\s*", comp_clean)
    return [p for b in blocks if (p := _parse_block(b)) is not None]


# Apply
df["ingredients_json"]  = df["composition_clean"].apply(
    lambda x: json.dumps(parse_composition(x), ensure_ascii=False)
)
df["ingredient_count"]  = df["ingredients_json"].apply(lambda x: len(json.loads(x)))
log.info("Ingredient extraction done.")
df[["composition_clean", "ingredients_json"]].head(8)

INFO | Ingredient extraction done.


,composition_clean,ingredients_json
0,Telmisartan 40mg + Hydrochlorothiazide 12.5mg,"[{""ingredient"": ""Telmisartan"", ""strength"": 40...."
1,Rosuvastatin 10mg,"[{""ingredient"": ""Rosuvastatin"", ""strength"": 10..."
2,Aceclofenac 100mg + Paracetamol 325mg,"[{""ingredient"": ""Aceclofenac"", ""strength"": 100..."
3,Cefpodoxime Proxetil 200mg + Clavulanic Acid 1...,"[{""ingredient"": ""Cefpodoxime Proxetil"", ""stren..."
4,Rabeprazole 20mg + Itopride 150mg,"[{""ingredient"": ""Rabeprazole"", ""strength"": 20...."
5,Levosulpiride 75mg + Rabeprazole 20mg,"[{""ingredient"": ""Levosulpiride"", ""strength"": 7..."
6,Glimepiride 2mg + Metformin 500mg,"[{""ingredient"": ""Glimepiride"", ""strength"": 2.0..."
7,Chlorpheniramine Maleate 2mg / 5ml + Paracetam...,"[{""ingredient"": ""Chlorpheniramine Maleate"", ""s..."


## 8 · Parser Self-Test

Must all pass before proceeding to bulk processing.

In [9]:
PARSER_TESTS = [
    ("Paracetamol 500mg",
     [{"ingredient":"Paracetamol","strength":500.0,"unit":"mg"}]),
    ("Paracetamol 500 mg",
     [{"ingredient":"Paracetamol","strength":500.0,"unit":"mg"}]),
    ("Cefpodoxime Proxetil 200mg",
     [{"ingredient":"Cefpodoxime Proxetil","strength":200.0,"unit":"mg"}]),
    ("Amoxicillin 500mg + Clavulanic Acid 125mg",
     [{"ingredient":"Amoxicillin","strength":500.0,"unit":"mg"},
      {"ingredient":"Clavulanic Acid","strength":125.0,"unit":"mg"}]),
    ("Vitamin C / Zinc",
     [{"ingredient":"Vitamin C","strength":None,"unit":""},
      {"ingredient":"Zinc","strength":None,"unit":""}]),
    ("Atorvastatin 10mg",
     [{"ingredient":"Atorvastatin","strength":10.0,"unit":"mg"}]),
    ("Metformin Hydrochloride 500mg",
     [{"ingredient":"Metformin Hydrochloride","strength":500.0,"unit":"mg"}]),
    ("Salbutamol Sulphate 100mcg",
     [{"ingredient":"Salbutamol Sulphate","strength":100.0,"unit":"mcg"}]),
]

passed = 0
for comp, expected in PARSER_TESTS:
    cleaned = clean_composition(comp)
    got     = parse_composition(cleaned)
    ok = (len(got) == len(expected) and all(
        g["ingredient"] == e["ingredient"] and g["strength"] == e["strength"]
        for g, e in zip(got, expected)
    ))
    if ok: passed += 1
    status = "✓ PASS" if ok else "✗ FAIL"
    print(f"{status}  |  {comp}")
    if not ok:
        print(f"         Expected : {expected}")
        print(f"         Got      : {got}")

print(f"\n{'✅' if passed==len(PARSER_TESTS) else '❌'}  {passed}/{len(PARSER_TESTS)} tests passed")
if passed < len(PARSER_TESTS):
    print("⚠  Fix failing cases in _parse_block before running bulk extraction.")

✓ PASS  |  Paracetamol 500mg
✓ PASS  |  Paracetamol 500 mg
✓ PASS  |  Cefpodoxime Proxetil 200mg
✓ PASS  |  Amoxicillin 500mg + Clavulanic Acid 125mg
✓ PASS  |  Vitamin C / Zinc
✓ PASS  |  Atorvastatin 10mg
✓ PASS  |  Metformin Hydrochloride 500mg
✓ PASS  |  Salbutamol Sulphate 100mcg

✅  8/8 tests passed


## 9 · Generic Name Generation

In [10]:
def build_generic_name(ingredients_json: str) -> str:
    try: ingredients = json.loads(ingredients_json)
    except: return ""
    names = [i["ingredient"].strip() for i in ingredients if i.get("ingredient")]
    return " + ".join(names)

def build_strength_summary(ingredients_json: str) -> str:
    try: ingredients = json.loads(ingredients_json)
    except: return ""
    parts = []
    for i in ingredients:
        nm = i.get("ingredient","").strip()
        st = i.get("strength")
        ut = i.get("unit","").strip()
        parts.append(f"{nm} {st}{ut}" if st is not None else nm)
    return " + ".join(parts)

df["generic_name"]     = df["ingredients_json"].apply(build_generic_name)
df["strength_summary"] = df["ingredients_json"].apply(build_strength_summary)
df[["brand_name" if "brand_name" in df.columns else name_col,
    "generic_name","strength_summary"]].head(10) if name_col else df[["generic_name","strength_summary"]].head(10)

,name,generic_name,strength_summary
0,Cardiomart H 40mg/12.5mg Tablet,Telmisartan + Hydrochlorothiazide,Telmisartan 40.0mg + Hydrochlorothiazide 12.5mg
1,Rosumart 10mg Tablet,Rosuvastatin,Rosuvastatin 10.0mg
2,Tendonil P 100mg/325mg Tablet,Aceclofenac + Paracetamol,Aceclofenac 100.0mg + Paracetamol 325.0mg
3,Xenoten CV 200mg/125mg Tablet,Cefpodoxime Proxetil + Clavulanic Acid,Cefpodoxime Proxetil 200.0mg + Clavulanic Acid...
4,Zomart IT 20mg/150mg Tablet SR,Rabeprazole + Itopride,Rabeprazole 20.0mg + Itopride 150.0mg
5,Zomart LS 75mg/20mg Tablet SR,Levosulpiride + Rabeprazole,Levosulpiride 75.0mg + Rabeprazole 20.0mg
6,Endomart G 2mg/500mg Tablet SR,Glimepiride + Metformin,Glimepiride 2.0mg + Metformin 500.0mg
7,Histadex P Syrup,Chlorpheniramine Maleate + 5ml + Paracetamol +...,Chlorpheniramine Maleate 2.0mg + 5ml + Paracet...
8,Histadex Syrup,Phenylephrine + 5ml + Chlorpheniramine Maleate...,Phenylephrine 5.0mg + 5ml + Chlorpheniramine M...
9,Tenospas 80mg/250mg Tablet,Drotaverine + Mefenamic Acid,Drotaverine 80.0mg + Mefenamic Acid 250.0mg


## 10 · Searchable & OCR Text Generation

In [11]:
def build_searchable_text(row) -> str:
    parts = [
        str(row.get(name_col, "") or ""),
        str(row.get("generic_name", "") or ""),
        str(row.get("marketer", "") or ""),
        str(row.get("uses", "") or "")[:400],
    ]
    return " | ".join(p.strip() for p in parts if p.strip()).lower()

def build_ocr_text(row) -> str:
    """Sorted, deduped token stream — maximises fuzzy OCR match recall."""
    raw = f"{row.get(name_col,'') or ''} {row.get('generic_name','') or ''}"
    raw = re.sub(r"[^a-zA-Z0-9 ]", " ", raw)
    tokens = sorted(set(t.lower() for t in raw.split() if len(t) > 1))
    return " ".join(tokens)

def build_embeddings_text(row) -> str:
    """Rich natural-language sentence for vector embedding."""
    return (
        f"Medicine: {row.get(name_col, '')}. "
        f"Generic composition: {row.get('generic_name', '')}. "
        f"Formulation: {row.get(comp_col, '') or ''}. "
        f"Uses: {str(row.get('uses', '') or '')[:600]}"
    )

df["searchable_text"] = df.apply(build_searchable_text, axis=1)
df["ocr_search_text"] = df.apply(build_ocr_text, axis=1)
df["embeddings_text"] = df.apply(build_embeddings_text, axis=1)

log.info("Search fields generated.")
df[["searchable_text","ocr_search_text"]].head(5)

INFO | Search fields generated.


,searchable_text,ocr_search_text
0,cardiomart h 40mg/12.5mg tablet | telmisartan ...,12 40mg 5mg cardiomart hydrochlorothiazide tab...
1,rosumart 10mg tablet | rosuvastatin | 10 drug ...,10mg rosumart rosuvastatin tablet
2,tendonil p 100mg/325mg tablet | aceclofenac + ...,100mg 325mg aceclofenac paracetamol tablet ten...
3,xenoten cv 200mg/125mg tablet | cefpodoxime pr...,125mg 200mg acid cefpodoxime clavulanic cv pro...
4,zomart it 20mg/150mg tablet sr | rabeprazole +...,150mg 20mg it itopride rabeprazole sr tablet z...


## 11 · medicine_id Assignment & Column Assembly

In [12]:
def make_medicine_id(url) -> str:
    if isinstance(url, str) and url.strip():
        return str(uuid.uuid5(uuid.NAMESPACE_URL, url.strip()))
    return str(uuid.uuid4())

df["medicine_id"]           = (df[url_col].apply(make_medicine_id)
                                if url_col else [str(uuid.uuid4()) for _ in range(len(df))])
df["brand_name"]            = df[name_col].str.strip()            if name_col else ""
df["normalized_brand_name"] = (df["brand_name"]
                                .str.lower()
                                .str.replace(r"[^a-z0-9 ]", " ", regex=True)
                                .str.strip())

WANT = [
    "medicine_id","brand_name","normalized_brand_name","marketer",
    comp_col,"composition_clean","ingredients_json","generic_name",
    "ingredient_count","strength_summary",
    "uses","side_effects",
    "searchable_text","ocr_search_text","embeddings_text","image_url",
]
FINAL_COLS = [c for c in WANT if c and c in df.columns]
df_final = df[FINAL_COLS].copy()

log.info("Final dataset: %d rows × %d columns", *df_final.shape)
df_final.head(3)

INFO | Final dataset: 223644 rows × 16 columns


,medicine_id,brand_name,normalized_brand_name,marketer,composition,composition_clean,ingredients_json,generic_name,ingredient_count,strength_summary,uses,side_effects,searchable_text,ocr_search_text,embeddings_text,image_url
0,7c29c08c-18ac-570c-afb1-57a417c2be5a,Cardiomart H 40mg/12.5mg Tablet,cardiomart h 40mg 12 5mg tablet,10 Drug Mart,Telmisartan (40mg) + Hydrochlorothiazide (12.5mg),Telmisartan 40mg + Hydrochlorothiazide 12.5mg,"[{""ingredient"": ""Telmisartan"", ""strength"": 40....",Telmisartan + Hydrochlorothiazide,2,Telmisartan 40.0mg + Hydrochlorothiazide 12.5mg,Treatment of Hypertension (high blood pressure...,Most side effects do not require any medical a...,cardiomart h 40mg/12.5mg tablet | telmisartan ...,12 40mg 5mg cardiomart hydrochlorothiazide tab...,Medicine: Cardiomart H 40mg/12.5mg Tablet. Gen...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow..."
1,170ed007-85ae-5ca5-af26-85a84e8471f4,Rosumart 10mg Tablet,rosumart 10mg tablet,10 Drug Mart,Rosuvastatin (10mg),Rosuvastatin 10mg,"[{""ingredient"": ""Rosuvastatin"", ""strength"": 10...",Rosuvastatin,1,Rosuvastatin 10.0mg,Treatment of High cholesterol Treatment of Hig...,Most side effects do not require any medical a...,rosumart 10mg tablet | rosuvastatin | 10 drug ...,10mg rosumart rosuvastatin tablet,Medicine: Rosumart 10mg Tablet. Generic compos...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow..."
2,e4911311-0468-5d90-be92-71c2a85b5cbf,Tendonil P 100mg/325mg Tablet,tendonil p 100mg 325mg tablet,10 Drug Mart,Aceclofenac (100mg) + Paracetamol (325mg),Aceclofenac 100mg + Paracetamol 325mg,"[{""ingredient"": ""Aceclofenac"", ""strength"": 100...",Aceclofenac + Paracetamol,2,Aceclofenac 100.0mg + Paracetamol 325.0mg,Pain relief In Pain relief Tendonil P 100mg/32...,Most side effects do not require any medical a...,tendonil p 100mg/325mg tablet | aceclofenac + ...,100mg 325mg aceclofenac paracetamol tablet ten...,Medicine: Tendonil P 100mg/325mg Tablet. Gener...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow..."


## 12 · Validation Rules

In [13]:
def validate_record(row) -> list[str]:
    flags = []
    try:    ingredients = json.loads(row["ingredients_json"])
    except: ingredients = []; flags.append("INVALID_JSON")

    if not ingredients:
        flags.append("EMPTY_INGREDIENTS")

    for ing in ingredients:
        nm   = str(ing.get("ingredient","")).strip()
        unit = str(ing.get("unit","")).strip()

        if not nm:
            flags.append("BLANK_INGREDIENT_NAME")
        elif nm.lower() in TRUNCATED_FLAGS:
            flags.append(f"SUSPICIOUS_INGREDIENT:{nm}")
        if len(nm.split()) == 1 and len(nm) <= 3 and nm.isalpha():
            flags.append(f"TOO_SHORT_INGREDIENT:{nm}")
        if unit and unit not in VALID_UNITS:
            flags.append(f"UNKNOWN_UNIT:{unit}")
        if ing.get("flag") == "SUSPICIOUS_TRUNCATION":
            flags.append(f"PARSER_FLAG:{nm}")

    if not row.get("generic_name",""):
        flags.append("EMPTY_GENERIC_NAME")
    if row.get("ingredient_count", 0) == 0:
        flags.append("ZERO_INGREDIENT_COUNT")
    if row.get("ingredient_count", 0) > 10:
        flags.append("HIGH_INGREDIENT_COUNT")

    return flags

df_final["validation_flags"] = df_final.apply(validate_record, axis=1)
df_final["flag_count"]       = df_final["validation_flags"].apply(len)
df_final["is_valid"]         = df_final["flag_count"] == 0

valid_pct = df_final["is_valid"].mean() * 100
log.info("Parsing accuracy (zero-flag records): %.2f%%", valid_pct)
print(f"Total records  : {len(df_final):>10,}")
print(f"Valid records  : {df_final['is_valid'].sum():>10,}  ({valid_pct:.2f}%)")
print(f"Invalid records: {(~df_final['is_valid']).sum():>10,}  ({100-valid_pct:.2f}%)")

INFO | Parsing accuracy (zero-flag records): 93.44%


Total records  :    223,644
Valid records  :    208,962  (93.44%)
Invalid records:     14,682  (6.56%)


## 13 · Failure Analysis

In [14]:
all_flags   = [f for flags in df_final["validation_flags"] for f in flags]
flag_counts = Counter(all_flags)

print("Top validation failures:\n")
for flag, cnt in flag_counts.most_common(25):
    bar = "█" * min(40, cnt // max(1, len(df_final)//100))
    print(f"  {cnt:8,}  {flag:40s}  {bar}")

df_suspicious = df_final[~df_final["is_valid"]].copy()
df_suspicious["flags_str"] = df_suspicious["validation_flags"].apply("; ".join)

show_cols = ["brand_name","composition_clean","flags_str"]
show_cols = [c for c in show_cols if c in df_suspicious.columns]
print("\nSample failed records:")
df_suspicious[show_cols].head(20)

Top validation failures:

    14,322  TOO_SHORT_INGREDIENT:w                    ██████
     8,627  TOO_SHORT_INGREDIENT:v                    ███
     1,278  TOO_SHORT_INGREDIENT:ml                   
       351  UNKNOWN_UNIT:AU                           
        87  HIGH_INGREDIENT_COUNT                     
        16  TOO_SHORT_INGREDIENT:gm                   
        13  UNKNOWN_UNIT:i.u                          
         8  UNKNOWN_UNIT:LF                           
         5  UNKNOWN_UNIT:CU                           
         3  UNKNOWN_UNIT:PFU                          
         3  UNKNOWN_UNIT:OU                           
         3  UNKNOWN_UNIT:Millions                     
         3  UNKNOWN_UNIT:mspores                      
         2  UNKNOWN_UNIT:millionspores                
         2  UNKNOWN_UNIT:10gm                         
         2  UNKNOWN_UNIT:ppm                          
         2  UNKNOWN_UNIT:miu                          
         2  UNKNOWN_UNIT:FFU  

,brand_name,composition_clean,flags_str
31,Momran Ointment,Hydroquinone 2% w / w + Mometasone 0.1% w / w ...,TOO_SHORT_INGREDIENT:w; TOO_SHORT_INGREDIENT:w...
118,Lulibiz 1% Cream,Luliconazole 1% w / w,TOO_SHORT_INGREDIENT:w
209,Danoff Lotion,Ketoconazole 2% w / v + Zinc pyrithione 1% w / v,TOO_SHORT_INGREDIENT:v; TOO_SHORT_INGREDIENT:v
210,Imuski 10ml Injection,Ranitidine 25mg / ml,TOO_SHORT_INGREDIENT:ml
211,Imuski Injection,Ranitidine 25mg / ml,TOO_SHORT_INGREDIENT:ml
212,Isedaz Injection,Midazolam 1mg / ml,TOO_SHORT_INGREDIENT:ml
214,Iondan Injection,Ondansetron 2mg / ml,TOO_SHORT_INGREDIENT:ml
215,6Ipzocin 30mg/ml Injection,Pentazocine 30mg / ml,TOO_SHORT_INGREDIENT:ml
218,Iket Injection,Ketamine 50mg / ml,TOO_SHORT_INGREDIENT:ml
303,Gabain Ointment,Capsaicin 0.025% w / w + Diclofenac 4% w / w +...,TOO_SHORT_INGREDIENT:w; TOO_SHORT_INGREDIENT:w...


## 14 · Parsing Accuracy Metrics

In [15]:
all_ingredients = []
for row_json in df_final["ingredients_json"]:
    try:
        all_ingredients.extend(json.loads(row_json))
    except: pass

unique_ingredients = len({i["ingredient"] for i in all_ingredients if i.get("ingredient")})

metrics = {
    "total_records":            int(len(df_final)),
    "valid_records":            int(df_final["is_valid"].sum()),
    "invalid_records":          int((~df_final["is_valid"]).sum()),
    "parsing_accuracy_pct":     round(valid_pct, 4),
    "unique_ingredients":       unique_ingredients,
    "avg_ingredients_per_med":  round(df_final["ingredient_count"].mean(), 3),
    "max_ingredients_in_one":   int(df_final["ingredient_count"].max()),
    "top_flags":                dict(flag_counts.most_common(15)),
}

print(json.dumps(metrics, indent=2))

out_report = REPORTS / "medicine_validation_report.json"
out_report.parent.mkdir(exist_ok=True)
out_report.write_text(json.dumps(metrics, indent=2))
log.info("Validation report → %s", out_report)

INFO | Validation report → c:\Users\arjun\OneDrive\Desktop\project\Medi Cam\reports\medicine_validation_report.json


{
  "total_records": 223644,
  "valid_records": 208962,
  "invalid_records": 14682,
  "parsing_accuracy_pct": 93.4351,
  "unique_ingredients": 3529,
  "avg_ingredients_per_med": 1.895,
  "max_ingredients_in_one": 19,
  "top_flags": {
    "TOO_SHORT_INGREDIENT:w": 14322,
    "TOO_SHORT_INGREDIENT:v": 8627,
    "TOO_SHORT_INGREDIENT:ml": 1278,
    "UNKNOWN_UNIT:AU": 351,
    "HIGH_INGREDIENT_COUNT": 87,
    "TOO_SHORT_INGREDIENT:gm": 16,
    "UNKNOWN_UNIT:i.u": 13,
    "UNKNOWN_UNIT:LF": 8,
    "UNKNOWN_UNIT:CU": 5,
    "UNKNOWN_UNIT:PFU": 3,
    "UNKNOWN_UNIT:OU": 3,
    "UNKNOWN_UNIT:Millions": 3,
    "UNKNOWN_UNIT:mspores": 3,
    "UNKNOWN_UNIT:millionspores": 2,
    "UNKNOWN_UNIT:10gm": 2
  }
}


## 15 · Ingredient Master Table

In [16]:
records = []
for _, row in df_final.iterrows():
    try:    ings = json.loads(row["ingredients_json"])
    except: continue
    for ing in ings:
        records.append({
            "medicine_id": row["medicine_id"],
            "brand_name":  row.get("brand_name",""),
            "ingredient":  ing.get("ingredient",""),
            "strength":    ing.get("strength"),
            "unit":        ing.get("unit",""),
            "flag":        ing.get("flag"),
        })

df_ingredients = pd.DataFrame(records)
log.info("Ingredient master: %d rows | %d unique ingredients",
         len(df_ingredients), df_ingredients["ingredient"].nunique())
df_ingredients.head(15)

INFO | Ingredient master: 423820 rows | 3529 unique ingredients


,medicine_id,brand_name,ingredient,strength,unit,flag
0,7c29c08c-18ac-570c-afb1-57a417c2be5a,Cardiomart H 40mg/12.5mg Tablet,Telmisartan,40.0,mg,None
1,7c29c08c-18ac-570c-afb1-57a417c2be5a,Cardiomart H 40mg/12.5mg Tablet,Hydrochlorothiazide,12.5,mg,None
2,170ed007-85ae-5ca5-af26-85a84e8471f4,Rosumart 10mg Tablet,Rosuvastatin,10.0,mg,None
3,e4911311-0468-5d90-be92-71c2a85b5cbf,Tendonil P 100mg/325mg Tablet,Aceclofenac,100.0,mg,None
4,e4911311-0468-5d90-be92-71c2a85b5cbf,Tendonil P 100mg/325mg Tablet,Paracetamol,325.0,mg,None
5,d497b86b-9b8e-5d6a-aecf-59b174376cb8,Xenoten CV 200mg/125mg Tablet,Cefpodoxime Proxetil,200.0,mg,None
6,d497b86b-9b8e-5d6a-aecf-59b174376cb8,Xenoten CV 200mg/125mg Tablet,Clavulanic Acid,125.0,mg,None
7,a8b27012-23b1-5648-b3fd-85edcba1941e,Zomart IT 20mg/150mg Tablet SR,Rabeprazole,20.0,mg,None
8,a8b27012-23b1-5648-b3fd-85edcba1941e,Zomart IT 20mg/150mg Tablet SR,Itopride,150.0,mg,None
9,6d128249-3267-5229-b3fa-c5e32c22df74,Zomart LS 75mg/20mg Tablet SR,Levosulpiride,75.0,mg,None


## 16 · Export Final Outputs

In [17]:
exports = {
    "final_clean_medicine_dataset.csv":
        df_final.drop(columns=["validation_flags"], errors="ignore"),
    "ingredient_master_table.csv":
        df_ingredients,
    "parsing_failure_samples.csv":
        df_suspicious[[c for c in ["brand_name","composition_clean","flags_str"]
                        if c in df_suspicious.columns]].head(5000),
    "suspicious_records.csv":
        df_suspicious.drop(columns=["validation_flags","embeddings_text"], errors="ignore"),
}

for filename, frame in exports.items():
    path = DATA_FINAL / filename
    frame.to_csv(path, index=False)
    log.info("✓ Saved %-45s  %d rows", filename, len(frame))

print("\n✅ All outputs exported.")

INFO | ✓ Saved final_clean_medicine_dataset.csv               223644 rows
INFO | ✓ Saved ingredient_master_table.csv                    423820 rows
INFO | ✓ Saved parsing_failure_samples.csv                    5000 rows
INFO | ✓ Saved suspicious_records.csv                         14682 rows



✅ All outputs exported.


## 17 · Next Steps

| Priority | Action | Condition |
|---|---|---|
| 🔴 **P0** | Inspect `parsing_failure_samples.csv` — fix `_parse_block` edge cases | Before any production use |
| 🔴 **P0** | Target ≥ 95 % zero-flag accuracy on full 226k dataset | Before promoting to `src/` |
| 🟡 **P1** | Promote helpers into `src/parsing/`, `src/normalization/` modules | After accuracy target met |
| 🟡 **P1** | Load `final_clean_medicine_dataset.csv` into Meilisearch for fuzzy search | After P0 complete |
| 🟢 **P2** | Run `embeddings_text` through `sentence-transformers` → pgvector | After P1 |
| 🟢 **P2** | Build FastAPI `/search`, `/autocomplete`, `/ocr-match` endpoints | After P1 |

### Recommended Production Structure (post-stabilisation)
```
src/
├── parsing/        ingredient_parser.py
├── normalization/  unit_normalizer.py  · composition_cleaner.py
├── validation/     record_validator.py · accuracy_reporter.py
├── search/         searchable_text.py  · ocr_tokenizer.py
└── export/         exporter.py
```